# Resumidor de commits v2 — Transformer + SentencePiece + 2 datasets

**Mejoras sobre v1** (BLEU 6.68, GRU+atencion, solo NNGen):
1. **Arquitectura**: Transformer encoder-decoder desde cero (sigue cumpliendo el limite: sin pre-entrenados)
2. **Tokenizador**: SentencePiece (subwords) — elimina `<unk>` y permite componer identificadores
3. **Datos**: NNGen (~27k, Java) + **CommitChronicle** (JetBrains Research, ASE 2023; 10.7M commits / 20 lenguajes, muestreados ~300k) → robustez multi-lenguaje.
   *Nota: se planeo CommitBERT (Jung 2021) pero sus enlaces de descarga ya no existen (404 verificado); CommitChronicle lo reemplaza con ventaja (mas grande, multi-lenguaje, HuggingFace).*
4. **Anti-generico (datos)**: filtrado de mensajes triviales/bots + label smoothing
5. **Limpieza del diff**: sin cabeceras git, nombre de archivo como token estructural
6. `MAX_DIFF` 100 → **256**

**Metodologia clave**: el TEST sigue siendo el de NNGen, intacto — asi v2 es comparable con v1 y con la literatura.

> ⚠️ **Tiempos**: preparacion ~30-40 min (el stream de CommitChronicle descarga varios GB) + entrenamiento **2-3 h en T4**. Sesion fresca de Colab con GPU T4.

In [ ]:
# ── 1. Dependencias, semillas, GPU ──────────────────────────────────────────
!pip -q install sentencepiece sacrebleu rouge-score

import json, os, re, random, zipfile, glob, math, time, hashlib
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import sentencepiece as spm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

assert torch.cuda.is_available(), 'Activa la GPU T4'
DEVICE = torch.device('cuda')
print('GPU:', torch.cuda.get_device_name(0), '| torch', torch.__version__)

In [ ]:
# ── 2. Datasets: NNGen + CommitBERT ─────────────────────────────────────────
!git clone -q --depth 1 https://github.com/Tbabm/nngen.git 2>/dev/null || echo 'nngen ya clonado'
!git clone -q --depth 1 https://github.com/graykode/commit-autosuggestions.git 2>/dev/null || echo 'commitbert ya clonado'

# Inspeccion del repo de CommitBERT ANTES de asumir su formato
for raiz, dirs, archivos in os.walk('commit-autosuggestions'):
    if '.git' in raiz:
        continue
    nivel = raiz.count(os.sep)
    if nivel <= 2:
        print(raiz + '/')
        for a in archivos[:10]:
            print('   ', a)

# El README del repo indica como descargar el dataset (suele ser via gdown).
print('\n────── README (secciones de dataset) ──────')
with open('commit-autosuggestions/README.md', encoding='utf-8', errors='replace') as f:
    texto = f.read()
for bloque in texto.split('\n#'):
    if 'ataset' in bloque[:100] or 'ownload' in bloque[:100]:
        print('#' + bloque[:1500])

In [ ]:
# ── 3. Carga de ambos corpus ────────────────────────────────────────────────
# NNGen: archivos paralelos .diff/.msg (formato conocido).
def cargar_nngen(split):
    d = next(iter(glob.glob(f'nngen/data/**/*{split}*.diff', recursive=True)))
    m = next(iter(glob.glob(f'nngen/data/**/*{split}*.msg', recursive=True)))
    with open(d, encoding='utf-8', errors='replace') as f:
        diffs = [l.strip() for l in f]
    with open(m, encoding='utf-8', errors='replace') as f:
        msgs = [l.strip() for l in f]
    assert len(diffs) == len(msgs)
    return list(zip(diffs, msgs))

nngen_train = cargar_nngen('train')
nngen_valid = cargar_nngen('valid')
nngen_test = cargar_nngen('test')
print(f'NNGen: {len(nngen_train)}/{len(nngen_valid)}/{len(nngen_test)}')

# ── Segundo dataset: CommitChronicle (JetBrains Research, ASE 2023) ─────────
# NOTA: originalmente se planeo CommitBERT (Jung 2021), pero sus carpetas de
# Google Drive ya no existen (404, verificado jul-2026). CommitChronicle es el
# reemplazo: 10.7M commits / 20 lenguajes / HuggingFace (descarga confiable).
# https://huggingface.co/datasets/JetBrains-Research/commit-chronicle
!pip -q install datasets

from datasets import load_dataset

OBJETIVO_CRUDO = 300_000   # ejemplos a tomar del stream (luego se filtran)
ds = load_dataset('JetBrains-Research/commit-chronicle', 'default',
                  split='train', streaming=True)
ds = ds.shuffle(seed=SEED, buffer_size=50_000)

commitbert_pares = []   # (se mantiene el nombre de variable para el resto del notebook)
primero = True
for ej in ds.take(OBJETIVO_CRUDO):
    if primero:
        print('CLAVES DEL EJEMPLO:', list(ej.keys()))
        primero = False
    msg = (ej.get('message') or '').strip()
    mods = ej.get('mods') or []
    partes = []
    for mod in mods[:3]:
        ruta = mod.get('new_path') or mod.get('old_path') or ''
        diff_mod = (mod.get('diff') or '')[:1200]
        if ruta:
            partes.append(f'<file> {ruta} </file>')
        partes.append(diff_mod)
    diff = ' '.join(partes).strip()
    if diff and msg:
        commitbert_pares.append((diff, msg))

print(f'CommitChronicle pares cargados: {len(commitbert_pares)}')
print('ejemplo:', commitbert_pares[0][1][:80], '||', commitbert_pares[0][0][:120])

In [ ]:
# ── 4. Limpieza y filtrado anti-generico ────────────────────────────────────
USAR_COMMITBERT = len(commitbert_pares) > 0
MAX_COMMITBERT = 180_000   # tope para caber en una sesion de Colab

RE_DIFF_HEADER = re.compile(r'\b(diff --git|index [0-9a-f]+\.\.[0-9a-f]+|new file mode \d+|deleted file mode \d+|@@[^@]*@@)')
RE_FILEPATH = re.compile(r'[ab]/([\w./-]+\.(java|py|js|go|rb|php|xml|md|txt|json|yml|yaml|gradle|properties))')
RE_ISSUE_ID = re.compile(r'\b[A-Z]{2,10}-\d+:?\s*')     # KAFKA-123, JIRA-9 ...
RE_ESPACIOS = re.compile(r'\s+')

GENERICOS = {'update', 'fix', 'wip', 'initial commit', 'first commit', 'test',
             'update readme', 'update readme . md', 'cleanup', 'minor fixes',
             'refactoring', 'refactor', 'changes', 'misc', 'typo', 'fix typo'}
RE_BOT = re.compile(r'merge (pull request|branch|remote)|dependabot|renovate|auto-generated', re.I)

def limpiar_diff(d):
    archivos = list(dict.fromkeys(m.group(1) for m in RE_FILEPATH.finditer(d)))[:3]
    d = RE_DIFF_HEADER.sub(' ', d)
    d = RE_ESPACIOS.sub(' ', d).strip()
    prefijo = ' '.join(f'<file> {a} </file>' for a in archivos)
    return (prefijo + ' ' + d).strip()

def limpiar_msg(m):
    m = RE_ISSUE_ID.sub('', m)          # quita ruido tipo KAFKA-123
    return RE_ESPACIOS.sub(' ', m).strip()

def filtrar(pares, es_test=False):
    vistos, salida = set(), []
    for d, m in pares:
        d2, m2 = limpiar_diff(d), limpiar_msg(m)
        if not es_test:
            nt = len(m2.split())
            if nt < 3 or nt > 32: continue
            if m2.lower() in GENERICOS: continue
            if RE_BOT.search(m2): continue
            if sum(c.isascii() for c in m2) / max(len(m2), 1) < 0.9: continue
            h = hashlib.md5((d2[:400]).encode()).hexdigest()
            if h in vistos: continue
            vistos.add(h)
        salida.append((d2, m2))
    return salida

train = filtrar(nngen_train)
if USAR_COMMITBERT:
    random.shuffle(commitbert_pares)
    train += filtrar(commitbert_pares[:MAX_COMMITBERT])
random.shuffle(train)
valid = filtrar(nngen_valid)
# TEST: limpieza SI, filtrado NO — debe seguir siendo el test integro de NNGen
test = filtrar(nngen_test, es_test=True)
print(f'train={len(train)}  valid={len(valid)}  test={len(test)} (test NNGen integro)')

In [ ]:
# ── 5. SentencePiece (vocabulario compartido 16k, subwords) ─────────────────
MAX_DIFF, MAX_MSG = 256, 32
VOCAB = 16_000
PAD, UNK, SOS, EOS = 0, 1, 2, 3

with open('corpus_spm.txt', 'w', encoding='utf-8') as f:
    for d, m in train:
        f.write(d[:2000] + '\n')
        f.write(m + '\n')

spm.SentencePieceTrainer.train(
    input='corpus_spm.txt', model_prefix='spm_v2', vocab_size=VOCAB,
    model_type='unigram', character_coverage=1.0,
    pad_id=PAD, unk_id=UNK, bos_id=SOS, eos_id=EOS,
    user_defined_symbols=['<file>', '</file>'],
    input_sentence_size=2_000_000, shuffle_input_sentence=True)

sp = spm.SentencePieceProcessor(model_file='spm_v2.model')
print('vocab:', sp.get_piece_size())
print('ejemplo:', sp.encode('Add UserValidator test class', out_type=str))

In [ ]:
# ── 6. Dataset y DataLoader ─────────────────────────────────────────────────
class Pares(Dataset):
    def __init__(self, pares):
        self.x = [sp.encode(d)[:MAX_DIFF] for d, _ in pares]
        self.y = [(sp.encode(m)[:MAX_MSG - 1] + [EOS]) for _, m in pares]
    def __len__(self):
        return len(self.x)
    def __getitem__(self, i):
        return self.x[i], self.y[i]

def colacionar(lote):
    xs, ys = zip(*lote)
    lx, ly = max(len(x) for x in xs), max(len(y) for y in ys)
    X = torch.full((len(xs), lx), PAD, dtype=torch.long)
    Y = torch.full((len(ys), ly), PAD, dtype=torch.long)
    for i, (x, y) in enumerate(zip(xs, ys)):
        X[i, :len(x)] = torch.tensor(x); Y[i, :len(y)] = torch.tensor(y)
    return X, Y

dl_train = DataLoader(Pares(train), batch_size=96, shuffle=True, collate_fn=colacionar, num_workers=2)
dl_valid = DataLoader(Pares(valid), batch_size=96, shuffle=False, collate_fn=colacionar)
print(len(dl_train), 'lotes/epoca')

In [ ]:
# ── 7. Transformer encoder-decoder (desde cero) ─────────────────────────────
D_MODEL, NHEAD, LAYERS, FF, DROPOUT = 256, 8, 4, 1024, 0.1

class PosEnc(nn.Module):
    def __init__(self, d, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d, 2).float() * (-math.log(10000.0) / d))
        pe[:, 0::2] = torch.sin(pos * div); pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class ResumidorV2(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(VOCAB, D_MODEL, padding_idx=PAD)   # compartida src/tgt
        self.pos = PosEnc(D_MODEL)
        self.tf = nn.Transformer(d_model=D_MODEL, nhead=NHEAD,
                                 num_encoder_layers=LAYERS, num_decoder_layers=LAYERS,
                                 dim_feedforward=FF, dropout=DROPOUT, batch_first=True)
        self.salida = nn.Linear(D_MODEL, VOCAB)
        self.salida.weight = self.emb.weight                        # tied embeddings
        self.escala = math.sqrt(D_MODEL)
    def forward(self, X, Y_in):
        mask_tgt = nn.Transformer.generate_square_subsequent_mask(Y_in.size(1)).to(X.device)
        out = self.tf(self.pos(self.emb(X) * self.escala), self.pos(self.emb(Y_in) * self.escala),
                      tgt_mask=mask_tgt,
                      src_key_padding_mask=(X == PAD), tgt_key_padding_mask=(Y_in == PAD),
                      memory_key_padding_mask=(X == PAD))
        return self.salida(out)

modelo = ResumidorV2().to(DEVICE)
print(f'Parametros: {sum(p.numel() for p in modelo.parameters())/1e6:.1f}M (desde cero)')

In [ ]:
# ── 8. Entrenamiento (label smoothing + warmup + mixed precision) ───────────
EPOCAS, PACIENCIA, WARMUP = 8, 2, 4000
perdida_fn = nn.CrossEntropyLoss(ignore_index=PAD, label_smoothing=0.1)  # anti-generico
opt = torch.optim.AdamW(modelo.parameters(), lr=5e-4, weight_decay=0.01)
scaler = torch.cuda.amp.GradScaler()
paso_global = 0

def lr_actual(paso):
    return min(paso / WARMUP, 1.0) * 5e-4

def correr_epoca(dl, entrenar):
    global paso_global
    modelo.train(entrenar)
    total, n = 0.0, 0
    for X, Y in dl:
        X, Y = X.to(DEVICE), Y.to(DEVICE)
        Y_in = torch.cat([torch.full((Y.size(0), 1), SOS, dtype=torch.long, device=DEVICE), Y[:, :-1]], dim=1)
        with torch.cuda.amp.autocast():
            logits = modelo(X, Y_in)
            perdida = perdida_fn(logits.reshape(-1, VOCAB), Y.reshape(-1))
        if entrenar:
            paso_global += 1
            for g in opt.param_groups:
                g['lr'] = lr_actual(paso_global)
            opt.zero_grad()
            scaler.scale(perdida).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(modelo.parameters(), 1.0)
            scaler.step(opt); scaler.update()
        total += perdida.item() * X.size(0); n += X.size(0)
    return total / n

mejor, sin_mejora = float('inf'), 0
for ep in range(1, EPOCAS + 1):
    t0 = time.time()
    tr = correr_epoca(dl_train, True)
    with torch.no_grad():
        va = correr_epoca(dl_valid, False)
    marca = ''
    if va < mejor:
        mejor, sin_mejora = va, 0
        torch.save(modelo.state_dict(), 'mejor_v2.pt'); marca = ' *guardado*'
    else:
        sin_mejora += 1
    print(f'epoca {ep} | train {tr:.3f} | val {va:.3f} | {(time.time()-t0)/60:.1f} min{marca}')
    if sin_mejora >= PACIENCIA:
        print('Parada temprana'); break

In [ ]:
# ── 9. Beam search (no-repeat bigram + penalizacion por longitud) ───────────
modelo.load_state_dict(torch.load('mejor_v2.pt')); modelo.eval()

@torch.no_grad()
def resumir(diff_texto, beam=5, alpha=0.7):
    ids = sp.encode(limpiar_diff(diff_texto))[:MAX_DIFF] or [UNK]
    X = torch.tensor([ids], device=DEVICE)
    haces = [(0.0, [SOS], False)]
    for _ in range(MAX_MSG):
        candidatos = []
        for lp, toks, fin in haces:
            if fin:
                candidatos.append((lp, toks, True)); continue
            Y_in = torch.tensor([toks], device=DEVICE)
            with torch.cuda.amp.autocast():
                logits = modelo(X, Y_in)[0, -1]
            logprobs = torch.log_softmax(logits.float(), dim=-1)
            bigramas = {(toks[i], toks[i+1]) for i in range(len(toks)-1)}
            top_lp, top_ix = logprobs.topk(beam * 3)
            agregados = 0
            for k in range(top_ix.numel()):
                if agregados >= beam: break
                t = int(top_ix[k])
                if len(toks) >= 1 and (toks[-1], t) in bigramas: continue
                candidatos.append((lp + float(top_lp[k]), toks + [t], t == EOS))
                agregados += 1
        haces = sorted(candidatos, key=lambda c: c[0] / (max(len(c[1]) - 1, 1) ** alpha), reverse=True)[:beam]
        if all(c[2] for c in haces): break
    mejor_toks = [t for t in haces[0][1] if t not in (PAD, SOS, EOS, UNK)]
    texto = sp.decode(mejor_toks)
    return re.sub(r'\s+', ' ', texto).strip()

# Ejemplos cualitativos, incluido el caso de la demo (UserValidator)
print('DEMO:', resumir('<file> UserValidator.java </file> new file + public class UserValidator { + public boolean isValidEmail ( String email ) { + return email != null && email . contains ( "@" ) ; } }'))
for i in random.sample(range(len(test)), 4):
    print('REAL:', test[i][1][:80])
    print('GEN :', resumir(test[i][0])[:80])
    print('─' * 60)

In [ ]:
# ── 10. Evaluacion en el TEST DE NNGEN (comparable con v1 y la literatura) ──
import sacrebleu
from rouge_score import rouge_scorer

print('Generando (10-15 min)...')
generados = [resumir(d) for d, _ in test]
referencias = [m for _, m in test]

bleu = sacrebleu.corpus_bleu(generados, [referencias])  # 13a: ambos en texto natural
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
rougeL = np.mean([scorer.score(r, g)['rougeL'].fmeasure for r, g in zip(referencias, generados)])
print(f'BLEU-4:  {bleu.score:.2f}   (v1: 6.68)')
print(f'ROUGE-L: {rougeL:.4f} (v1: 0.2092)')
# Nota de protocolo: v2 genera texto detokenizado (SentencePiece), por lo que
# se usa la tokenizacion 13a de sacrebleu sobre ambas partes. Documentar.

In [ ]:
# ── 11. Exportar artefactos v2 ──────────────────────────────────────────────
from google.colab import files
os.makedirs('artefactos_v2', exist_ok=True)

torch.save({'modelo': modelo.state_dict(),
            'hiperparametros': {'VOCAB': VOCAB, 'D_MODEL': D_MODEL, 'NHEAD': NHEAD,
                                 'LAYERS': LAYERS, 'FF': FF, 'MAX_DIFF': MAX_DIFF,
                                 'MAX_MSG': MAX_MSG}},
           'artefactos_v2/modelo_resumidor_v2.pt')
!cp spm_v2.model artefactos_v2/

metadatos = {
    'arquitectura': 'Transformer 4+4 capas, d_model 256, SentencePiece 16k compartido, desde cero',
    'datasets': ['github:Tbabm/nngen',
                 'hf:JetBrains-Research/commit-chronicle (ASE 2023, muestra ~300k; reemplaza a CommitBERT cuyos enlaces estan caidos)'],
    'train_final': len(train), 'bleu4_test': round(float(bleu.score), 2),
    'rougeL_test': round(float(rougeL), 4), 'torch': torch.__version__, 'seed': SEED,
    'nota_protocolo': 'test = NNGen integro; BLEU 13a sobre texto detokenizado',
}
with open('artefactos_v2/metricas_v2.json', 'w') as f:
    json.dump(metadatos, f, indent=2)

with zipfile.ZipFile('artefactos_resumidor_v2.zip', 'w') as z:
    for fn in os.listdir('artefactos_v2'):
        z.write(f'artefactos_v2/{fn}', fn)
files.download('artefactos_resumidor_v2.zip')
print('Guardar en Github-commit-summarizer-ms/training/artefactos_v2/ (NO pisar los v1)')